# Neural CFR+ on the 69-claim game

This notebook runs the first `>64`-claim neural CFR+ reference:

- `r6_s4_h4_hp2tfhq_ss` (69 claims);
- sampled traverser expansion capped at 16 claims, always retaining `CALL`;
- one atomically replaced full-resume checkpoint;
- lightweight playable average-policy snapshots every five measured training minutes.

The policy snapshots are what should be used for later fitted-return best-response evaluation. They are small. `latest_checkpoint.pt` is the only large persistent file and exists solely for exact resume.

In [ ]:
from datetime import datetime, timezone
import json
import math
from pathlib import Path
import shutil
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'liars_poker').is_dir():
    raise RuntimeError('Could not locate the liars_poker repository root.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.env import rules_for_spec
from liars_poker.eval.neural_evaluators import PolicySnapshotEvaluator, ScheduledEvaluation
from liars_poker.policies.neural import InfosetEncoder
from liars_poker.training.neural_runs import run_neural_cfr_plus

assert torch.cuda.is_available(), 'This reference run requires CUDA.'
device = 'cuda'
torch.set_float32_matmul_precision('high')
print('repository:', REPO_ROOT)
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('free / total GPU GiB:', tuple(round(x / 2**30, 2) for x in torch.cuda.mem_get_info()))

In [ ]:
spec = GameSpec(
    ranks=6,
    suits=4,
    hand_size=4,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips', 'FullHouse', 'Quads'),
    suit_symmetry=True,
)

TRAINING_MINUTES = 45
POLICY_SNAPSHOT_EVERY_MINUTES = 5
CHECKPOINT_EVERY_MINUTES = 10
TRAVERSALS_PER_PLAYER = 256
SEED = 7

# One million strategy records per player keeps the exact-resume checkpoint
# near 1.5 GiB rather than roughly 3 GiB at the old 2M/player setting.
STRATEGY_BUFFER_CAPACITY = 1_000_000

TRAINER_KWARGS = {
    'regret_hidden_sizes': (2048, 2048),
    'strategy_hidden_sizes': (512, 512),
    'device': device,
    'seed': SEED,
    'regret_buffer_capacity': 1_000_000,
    'strategy_buffer_capacity': STRATEGY_BUFFER_CAPACITY,
    'learning_rate': 1e-3,
    'batch_size': 4096,
    'regret_train_steps': 24,
    'strategy_train_steps': 6,
    'regret_positive_weight': 0.5,
    'strategy_weighting': 'linear',
    'validation_fraction': 0.02,
    'validation_buffer_capacity': 20_000,
    'traversal_backend': 'gpu_native',
    'traversal_batch_size': 64,
    'traverser_action_sample_count': 16,
    'traverser_action_sample_fraction': None,
    'traverser_action_full_first': False,
    'traverser_action_priority_count': 0,
    'traverser_action_baseline': 'none',
    'device_replay': True,
    'fused_optimizer': True,
    'amp_dtype': None,
    'compile_models': False,
}

rules = rules_for_spec(spec)
encoder = InfosetEncoder(spec)
assert len(rules.claims) == 69
print('spec:', spec.to_short_str())
print('claims:', len(rules.claims))
print('network input / output:', encoder.input_dim, encoder.action_dim)

## Disk preflight

CFR+ checkpoints do **not** save the transient regret buffers. The dominant checkpoint payload is the two average-strategy reservoirs. Atomic replacement temporarily requires both the old checkpoint and the new checkpoint to coexist.

In [ ]:
def mlp_parameter_count(input_dim, hidden_sizes, output_dim):
    sizes = (input_dim, *hidden_sizes, output_dim)
    return sum((left + 1) * right for left, right in zip(sizes, sizes[1:]))

bytes_per_replay_record = (
    4 * encoder.input_dim
    + 4 * encoder.action_dim
    + encoder.action_dim
    + 4
)
regret_params = mlp_parameter_count(
    encoder.input_dim,
    TRAINER_KWARGS['regret_hidden_sizes'],
    encoder.action_dim,
)
strategy_params = mlp_parameter_count(
    encoder.input_dim,
    TRAINER_KWARGS['strategy_hidden_sizes'],
    encoder.action_dim,
)

strategy_replay_bytes = 2 * STRATEGY_BUFFER_CAPACITY * bytes_per_replay_record
validation_replay_bytes = (
    2 * TRAINER_KWARGS['validation_buffer_capacity'] * bytes_per_replay_record
)
# Model parameters plus two Adam moments. This is an estimate, not a byte-exact serializer model.
model_optimizer_bytes = 12 * (2 * regret_params + 2 * strategy_params)
checkpoint_estimate_bytes = int(
    1.15 * (strategy_replay_bytes + validation_replay_bytes + model_optimizer_bytes)
)
policy_snapshot_estimate_bytes = 4 * 2 * strategy_params
expected_snapshots = math.ceil(TRAINING_MINUTES / POLICY_SNAPSHOT_EVERY_MINUTES)
free_bytes = shutil.disk_usage(REPO_ROOT).free
atomic_peak_estimate_bytes = 2 * checkpoint_estimate_bytes

storage = pd.Series({
    'bytes per replay record': bytes_per_replay_record,
    'estimated full checkpoint GiB': checkpoint_estimate_bytes / 2**30,
    'estimated atomic checkpoint peak GiB': atomic_peak_estimate_bytes / 2**30,
    'estimated policy snapshot MiB': policy_snapshot_estimate_bytes / 2**20,
    'estimated all scheduled snapshots MiB': (
        expected_snapshots * policy_snapshot_estimate_bytes / 2**20
    ),
    'currently free disk GiB': free_bytes / 2**30,
})
display(storage.to_frame('estimate').style.format(precision=3))

required_free = atomic_peak_estimate_bytes + 512 * 2**20
if free_bytes < required_free:
    raise RuntimeError(
        'Insufficient free disk for safe atomic checkpoint replacement. '
        f'Need about {required_free / 2**30:.2f} GiB; '
        f'currently have {free_bytes / 2**30:.2f} GiB.'
    )

In [ ]:
def policy_snapshot_schedule():
    # A new stateful schedule object is required for each fresh or resumed session.
    return ScheduledEvaluation(
        name='policy_snapshots',
        evaluator=PolicySnapshotEvaluator(),
        every_minutes=POLICY_SNAPSHOT_EVERY_MINUTES,
        run_at_end=True,
    )

run_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
RUN_DIR = (
    REPO_ROOT
    / 'artifacts'
    / 'cfr_plus_approx_reference_runs'
    / f'{spec.to_short_str()}___{run_id}'
)
ACTIVE_RUN_DIR = RUN_DIR
print('new run directory:', RUN_DIR)

## Fresh 45-minute run

Set `RUN_FRESH = True` only when ready. Measured training time excludes checkpoint and snapshot serialization.

In [ ]:
RUN_FRESH = False

if RUN_FRESH:
    fresh_result = run_neural_cfr_plus(
        spec,
        minutes=TRAINING_MINUTES,
        traversals_per_player=TRAVERSALS_PER_PLAYER,
        trainer_kwargs=TRAINER_KWARGS,
        evaluations=[policy_snapshot_schedule()],
        checkpoint_every_minutes=CHECKPOINT_EVERY_MINUTES,
        run_dir=RUN_DIR,
        save_checkpoint=True,
        wait_for_evaluations=True,
        debug=True,
    )
    ACTIVE_RUN_DIR = fresh_result.run_dir
    print('completed:', ACTIVE_RUN_DIR)

## Exact resume

Set `RESUME_RUN_DIR` to an existing run directory and choose the **additional** measured training minutes. The same `latest_checkpoint.pt` is replaced; historical policy snapshots remain available.

In [ ]:
RESUME_RUN_DIR = None  # Example: Path('/root/liars_poker/artifacts/.../r6_s4_h4_...')
ADDITIONAL_TRAINING_MINUTES = 15

if RESUME_RUN_DIR is not None:
    RESUME_RUN_DIR = Path(RESUME_RUN_DIR)
    resume_result = run_neural_cfr_plus(
        spec,
        minutes=ADDITIONAL_TRAINING_MINUTES,
        traversals_per_player=TRAVERSALS_PER_PLAYER,
        evaluations=[policy_snapshot_schedule()],
        checkpoint_every_minutes=CHECKPOINT_EVERY_MINUTES,
        resume_from=RESUME_RUN_DIR,
        device=device,
        save_checkpoint=True,
        wait_for_evaluations=True,
        debug=True,
    )
    ACTIVE_RUN_DIR = resume_result.run_dir
    print('resumed through measured minute:', resume_result.measured_training_s / 60.0)

## Run diagnostics and artifact audit

This cell confirms that there is one large resume checkpoint and lists the small policies available for later approximate-BR evaluation.

In [ ]:
def read_jsonl(path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line]

def directory_bytes(path):
    if not path.exists():
        return 0
    return sum(item.stat().st_size for item in path.rglob('*') if item.is_file())

ACTIVE_RUN_DIR = Path(ACTIVE_RUN_DIR)
manifest_path = ACTIVE_RUN_DIR / 'manifest.json'
training_path = ACTIVE_RUN_DIR / 'training.jsonl'
snapshot_root = ACTIVE_RUN_DIR / 'evaluations' / 'policy_snapshots' / 'snapshots'
checkpoint_path = ACTIVE_RUN_DIR / 'latest_checkpoint.pt'
checkpoint_temp_path = ACTIVE_RUN_DIR / 'latest_checkpoint.pt.new'
policy_dir = ACTIVE_RUN_DIR / 'policy'

manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else {}
training = pd.DataFrame(read_jsonl(training_path))
snapshot_dirs = sorted(path for path in snapshot_root.glob('*') if path.is_dir()) if snapshot_root.exists() else []

artifact_rows = [
    {'artifact': 'latest exact-resume checkpoint', 'count': int(checkpoint_path.exists()), 'GiB': (checkpoint_path.stat().st_size / 2**30 if checkpoint_path.exists() else 0.0)},
    {'artifact': 'incomplete checkpoint temporary', 'count': int(checkpoint_temp_path.exists()), 'GiB': (checkpoint_temp_path.stat().st_size / 2**30 if checkpoint_temp_path.exists() else 0.0)},
    {'artifact': 'playable evaluation snapshots', 'count': len(snapshot_dirs), 'GiB': sum(directory_bytes(path) for path in snapshot_dirs) / 2**30},
    {'artifact': 'final playable policy', 'count': int(policy_dir.exists()), 'GiB': directory_bytes(policy_dir) / 2**30},
]
display(pd.DataFrame(artifact_rows).set_index('artifact').style.format({'GiB': '{:.4f}'}))
print('status:', manifest.get('status'))
print('measured training min:', float(manifest.get('measured_training_s', 0.0)) / 60.0)

snapshot_table = []
for path in snapshot_dirs:
    result_path = path / 'result.json'
    result = json.loads(result_path.read_text(encoding='utf-8')) if result_path.exists() else {}
    snapshot_table.append({
        'training min': float(result.get('measured_training_s', 0.0)) / 60.0,
        'iteration': result.get('iteration'),
        'MiB': directory_bytes(path) / 2**20,
        'policy directory': str(path),
    })
if snapshot_table:
    display(pd.DataFrame(snapshot_table).style.format({'training min': '{:.2f}', 'MiB': '{:.2f}'}))

if not training.empty:
    timing = pd.json_normalize(training['timing'])
    x = training['measured_training_s'] / 60.0
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for column, label in [
        ('traversal_s', 'traversal'),
        ('regret_training_s', 'regret fit'),
        ('strategy_training_s', 'strategy fit'),
    ]:
        if column in timing:
            axes[0].plot(x, timing[column], alpha=0.75, label=label)
    axes[0].set(title='Per-iteration timing', xlabel='Measured training minutes', ylabel='Seconds')
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    strategy_sizes = pd.DataFrame(training['strategy_buffer_sizes'].tolist())
    strategy_seen = pd.DataFrame(training['strategy_records_seen'].tolist())
    for pid in range(strategy_sizes.shape[1]):
        axes[1].plot(
            x,
            strategy_sizes[pid] / strategy_seen[pid].clip(lower=1),
            label=f'player {pid + 1}',
        )
    axes[1].set(title='Strategy reservoir retained fraction', xlabel='Measured training minutes', ylabel='Fraction')
    axes[1].grid(alpha=0.3)
    axes[1].legend()
    plt.tight_layout()
    plt.show()

## Later evaluation

Each directory in `snapshot_table['policy directory']` is directly loadable by `run_best_responder`. Evaluate snapshots serially so responder replay and GPU memory are released between runs. Do not use `latest_checkpoint.pt` as an evaluation target: it is large trainer state, whereas the snapshot directories are ordinary playable policies.